# BreastMNIST: ResNet-50 weights V1 against V2

The BreastMNIST features come from ResNet-50 with torchvision's
`IMAGENET1K_V2` weights, its default and its more accurate set on ImageNet
(top-1 80.9% against 76.1% for V1, from torchvision's metadata below). On
FGVC-Aircraft, V2 failed: 80% of its features were exact zeros and its AUC
with all features was 0.742, against 0.862 for V1 (detailed report, section
2.3). This notebook asks whether BreastMNIST has the same problem.

The protocol is that of the FGVC-Aircraft comparison
(`scripts/aircraft_weights.py`) and of the main BreastMNIST runs:

- every non-constant feature, standardized on the training set;
- logistic regression, `max_iter=1000`, C = 1;
- 16 random splits, 70% of the images in the training set, seed 42;
- test AUC, with *malignant* as the positive class.

Both versions see the same splits, so their AUCs are compared split by
split with the corrected resampled t-test (Nadeau and Bengio 2003), as in
the detailed report. It runs twice: on all 780 images, and on the 585 left
without the copies (`demos/copies.ipynb`).

The first call for V1 embeds the 780 images (seconds on a GPU) and caches
them as `embeddings_resnet50_v1.npz`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from torchvision.models import ResNet50_Weights

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT))  # repo root, so featsel_datasets imports

from featsel import environment
from featsel.run import run
from featsel_datasets.medmnist_breastmnist.breastmnist import BreastMNIST
from featsel_datasets.medmnist_breastmnist.featsel_view import NAME, TARGET, load_xy

WEIGHTS = ('V1', 'V2')
IMAGE_SETS = {'all 780': False, 'without copies': True}  # name: drop_copies
CONFIG = {
    'seed': 42,
    'n_splits': 16,
    'train_sizes': [0.7],
    'preprocess': ['standard'],
    'selection_repeats': 1,
    'metrics': ['roc_auc'],
    'models': [{'name': 'logistic_regression', 'params': {'max_iter': 1000}}],
}
OUT = ROOT / 'results' / 'breastmnist_weights.csv'

pd.DataFrame({w: ResNet50_Weights[f'IMAGENET1K_{w}'].meta['_metrics']['ImageNet-1K']
              for w in WEIGHTS})

,V1,V2
acc@1,76.130,80.858
acc@5,92.862,95.434


## The features

The share of exact zeros among all 780 x 2,048 values, and the number of
features left after the constant ones are dropped.

In [2]:
data = {w: BreastMNIST(weights=w) for w in WEIGHTS}
pd.DataFrame({w: {'zero share': (d.embeddings.to_numpy() == 0).mean(),
                  'non-constant features': load_xy(d)[0].shape[1]}
              for w, d in data.items()}).T

,zero share,non-constant features
V1,0.027759,2048.0
V2,0.428191,2048.0


## AUC with all features

Random selection of every feature is the runner's all-features reference.
Each run's own CSV goes to `results/local/`.

In [3]:
frames = []
for image_set, drop_copies in IMAGE_SETS.items():
    for w, d in data.items():
        X, y = load_xy(d, drop_copies=drop_copies)
        tag = f"{w.lower()}{'_nocopies' if drop_copies else ''}"
        config = {**CONFIG,
                  'selectors': [{'name': 'RandomSelector', 'k': [X.shape[1]]}],
                  'output': str(ROOT / 'results' / 'local' / f'breastmnist_weights_{tag}.csv')}
        rows = pd.read_csv(run(X, y, config, target=TARGET, name=f'{NAME} {w} {image_set}'))
        frames.append(rows[['split', 'n_train', 'n_test', 'roc_auc']]
                      .assign(images=image_set, weights=w))

table = pd.concat(frames, ignore_index=True)
table.to_csv(OUT, index=False)
environment.write_json(OUT.with_suffix('.json'), config=CONFIG, weights=WEIGHTS,
                       image_sets=IMAGE_SETS)
table.groupby(['images', 'weights'])['roc_auc'].describe()[['count', 'mean', '25%', '50%', '75%']]


=== /home/dror/projects/featsel/results/local/breastmnist_weights_v1.csv ===
Dataset      : BreastMNIST ResNet-50 V1 all 780, 780 samples x 2048 features, target 'label'
Seed         : 42
Tasks        : 1
  binary                 benign_or_normal 570, malignant 210
Splits       : 16 stratified Monte Carlo draws
  train_size 0.7: 546 train / 234 test
Repeats      : 1 fits per selection, median time recorded
Preprocess   : standard
Metrics      : roc_auc
Workers      : 1 (parallel over task x selector jobs, one shared copy of each split)
Memory cap   : 8.9 GB of data per worker
Selectors    :
  RandomSelector   k = 2048 (100.00%)
Models       :
  logistic_regression {'max_iter': 1000}

Total        : 16 executions (1 tasks x 16 splits x 1 train sizes x 1 preprocess x 1 selector-k x 1 models)



  0%|          | 0/16 [00:00<?, ?exec/s]

Wrote 16 executions to /home/dror/projects/featsel/results/local/breastmnist_weights_v1.csv in 27s



=== /home/dror/projects/featsel/results/local/breastmnist_weights_v2.csv ===
Dataset      : BreastMNIST ResNet-50 V2 all 780, 780 samples x 2048 features, target 'label'
Seed         : 42
Tasks        : 1
  binary                 benign_or_normal 570, malignant 210
Splits       : 16 stratified Monte Carlo draws
  train_size 0.7: 546 train / 234 test
Repeats      : 1 fits per selection, median time recorded
Preprocess   : standard
Metrics      : roc_auc
Workers      : 1 (parallel over task x selector jobs, one shared copy of each split)
Memory cap   : 8.9 GB of data per worker
Selectors    :
  RandomSelector   k = 2048 (100.00%)
Models       :
  logistic_regression {'max_iter': 1000}

Total        : 16 executions (1 tasks x 16 splits x 1 train sizes x 1 preprocess x 1 selector-k x 1 models)



  0%|          | 0/16 [00:00<?, ?exec/s]

Wrote 16 executions to /home/dror/projects/featsel/results/local/breastmnist_weights_v2.csv in 17s


... (3 outputs left out)


=== /home/dror/projects/featsel/results/local/breastmnist_weights_v2_nocopies.csv ===
Dataset      : BreastMNIST ResNet-50 V2 without copies, 585 samples x 2048 features, target 'label'
Seed         : 42
Tasks        : 1
  binary                 benign_or_normal 405, malignant 180
Splits       : 16 stratified Monte Carlo draws
  train_size 0.7: 410 train / 175 test
Repeats      : 1 fits per selection, median time recorded
Preprocess   : standard
Metrics      : roc_auc
Workers      : 1 (parallel over task x selector jobs, one shared copy of each split)
Memory cap   : 8.9 GB of data per worker
Selectors    :
  RandomSelector   k = 2048 (100.00%)
Models       :
  logistic_regression {'max_iter': 1000}

Total        : 16 executions (1 tasks x 16 splits x 1 train sizes x 1 preprocess x 1 selector-k x 1 models)



  0%|          | 0/16 [00:00<?, ?exec/s]

Wrote 16 executions to /home/dror/projects/featsel/results/local/breastmnist_weights_v2_nocopies.csv in 12s


count      mean       25%       50%       75%
images         weights                                               
all 780        V1        16.0  0.897852  0.883830  0.895920  0.907071
               V2        16.0  0.923901  0.911585  0.923211  0.936253
without copies V1        16.0  0.875920  0.853066  0.873710  0.894126
               V2        16.0  0.894638  0.876821  0.894050  0.906041

## V1 minus V2, split by split

The corrected resampled t-test widens the variance by `n_test / n_train`,
because the training sets of different splits overlap.

In [4]:
def corrected_test(diff, n_test, n_train):
    """Mean difference, its 95% interval and p-value (Nadeau and Bengio 2003)."""
    j = len(diff)
    se = np.sqrt(diff.var(ddof=1) * (1 / j + n_test / n_train))
    half = stats.t.ppf(0.975, j - 1) * se
    p = 2 * stats.t.sf(abs(diff.mean() / se), j - 1)
    return {'V1 - V2': diff.mean(), 'low': diff.mean() - half, 'high': diff.mean() + half,
            'p': p, 'V1 wins': int((diff > 0).sum()), 'splits': j}


tests = {}
for image_set, part in table.groupby('images'):
    auc = part.pivot(index='split', columns='weights', values='roc_auc')
    sizes = part[part.weights == 'V1'].iloc[0]
    tests[image_set] = corrected_test(auc['V1'] - auc['V2'], sizes.n_test, sizes.n_train)
pd.DataFrame(tests).T

,V1 - V2,low,high,p,V1 wins,splits
all 780,-0.026049,-0.050631,-0.001466,0.039234,0.0,16.0
without copies,-0.018718,-0.061033,0.023597,0.360706,3.0,16.0


## Conclusion

On BreastMNIST, V2 is the better choice, unlike on FGVC-Aircraft.

- **All 780 images:** AUC 0.924 with V2 against 0.898 with V1. V2 wins all
  16 splits, and the difference is established (p = 0.039).
- **Without the copies:** 0.895 against 0.876. V2 wins 13 of 16 splits, but
  the difference is not established (p = 0.36).
- **Zeros:** 43% of V2's values are exact zeros, against 2.8% of V1's. On
  FGVC-Aircraft the share was 80%. Here the zeros did not hurt: no feature
  is constant with either version.

V2's AUC on all images equals the all-features result of the main runs
(detailed report, section 8.1), so these are the features they used.

The better weights depend on the dataset, so each dataset needs this check.
It uses test AUC, so it confirms V2, which was chosen before any result; it
is not a new search for the best weights.